# Week 6 Assignment: Price Band Classification Fine-tuning

This notebook implements an end-to-end workflow for fine-tuned price-band classification (budget, midrange, premium) using OpenAI models. It covers data loading, binning, dataset prep, fine-tuning, inference, and evaluation.

---

## 1. Import Required Libraries and Dependencies
Import all necessary Python packages for the project.

In [8]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE_URL="https://openrouter.ai/api/v1"
assert OPENAI_API_KEY, "OpenAI API key not found in environment variables!"


openai_client = OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL)


In [ ]:
# Utility function: Load dataset

def load_price_data(file_path):
    """Load price dataset from CSV/XLSX."""
    ext = os.path.splitext(file_path)[-1].lower()
    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format: {}".format(ext))
    return df

# Example usage (replace with actual file path)
df = load_price_data('products.csv')
df.head()


In [ ]:
# Utility function: Create price bins

def create_price_bins(df, price_col, bins=(0, 100, 500, np.inf), labels=("budget", "midrange", "premium")):
    """Add price band column to DataFrame."""
    df['price_band'] = pd.cut(df[price_col], bins=bins, labels=labels)
    return df

df = create_price_bins(df, 'price')
df['price_band'].value_counts()


In [ ]:
# Utility function: Visualize price distribution and class balance

def plot_price_distribution(df, price_col, band_col='price_band'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(df[price_col], bins=30, ax=axes[0])
    axes[0].set_title('Price Distribution')
    sns.countplot(x=band_col, data=df, ax=axes[1])
    axes[1].set_title('Price Band Class Balance')
    plt.tight_layout()
    plt.show()

# Example usage
plot_price_distribution(df, 'price')


In [ ]:
# Utility function: Prepare fine-tuning dataset

def prepare_finetune_dataset(df, text_col, band_col='price_band', output_path='finetune_data.jsonl'):
    """Export dataset for OpenAI fine-tuning (JSONL format)."""
    records = []
    for _, row in df.iterrows():
        records.append({
            "messages": [
                {"role": "system", "content": "Classify the product into price band."},
                {"role": "user", "content": str(row[text_col])},
                {"role": "assistant", "content": str(row[band_col])}
            ]
        })
    with open(output_path, 'w') as f:
        for rec in records:
            f.write(f"{rec}\n")
    return output_path

# Example usage
prepare_finetune_dataset(df, 'description')


In [ ]:
# Utility function: Upload file to OpenAI for fine-tuning

def upload_file_to_openai(file_path):
    with open(file_path, "rb") as f:
        resp = openai_client.files.create(file=f, purpose="fine-tune")
    return resp.id

# Example usage
file_id = upload_file_to_openai('finetune_data.jsonl')


In [ ]:
# Utility function: Create and monitor fine-tuning job

def create_finetune_job(file_id, model="gpt-3.5-turbo"):
    job = openai_client.fine_tuning.jobs.create(training_file=file_id, model=model)
    return job.id

# Example usage
job_id = create_finetune_job(file_id)


def monitor_finetune_job(job_id):
    job = openai_client.fine_tuning.jobs.retrieve(job_id)
    return job.status, job

# Example usage
status, job = monitor_finetune_job(job_id)


In [ ]:
# Utility function: Run inference with fine-tuned model

def classify_price_band(model_name, product_text):
    resp = openai_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "Classify the product into price band."},
            {"role": "user", "content": product_text}
        ]
    )
    return resp.choices[0].message.content.strip()

# Example usage
predicted_band = classify_price_band('ft:gpt-3.5-turbo:your-job-id', 'Product description here')


In [ ]:
# Utility function: Evaluate classification performance

def evaluate_classification(y_true, y_pred, labels=("budget", "midrange", "premium")):
    print("Classification Report:")
    print(classification_report(y_true, y_pred, labels=labels))
    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()
    f1 = f1_score(y_true, y_pred, average=None, labels=labels)
    plt.bar(labels, f1)
    plt.title('Per-class F1 Score')
    plt.ylabel('F1 Score')
    plt.show()
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")
    return cm, f1, acc

# Example usage
# cm, f1, acc = evaluate_classification(y_true, y_pred)


---

## Workflow Example

1. **Load dataset**
2. **Create price bins**
3. **Visualize price distribution and class balance**
4. **Prepare fine-tuning dataset**
5. **Upload file to OpenAI**
6. **Create and monitor fine-tuning job**
7. **Run inference with fine-tuned model**
8. **Evaluate classification performance**

Use the utility functions above for each step. Replace file paths, column names, and model names as needed for your dataset and workflow.